# Adaptive Supply Chain RL — GRPO Training Notebook

Train `Qwen2.5-7B-Instruct` via GRPO to manage perishable goods inventory, negotiate with a reactive supplier, and survive the day-21 supply disruption crisis.

The environment is **domain-agnostic** — the trained agent works for any perishable goods context (pharmaceuticals, food logistics, electronics components, etc.).

**Before running in Colab:** clone the repo and `cd` into it, or mount your Drive with the repo.
```bash
!git clone https://huggingface.co/spaces/<your-org>/asc-supply-chain
%cd asc-supply-chain
```

**Runtime:** GPU (T4 minimum, A100 recommended for 7B). Set `MODEL_NAME` to `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` if VRAM is tight.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q unsloth trl transformers accelerate pydantic requests matplotlib datasets

In [ ]:
# Cell 2 — Config
import os

ENV_URL      = os.getenv("SUPPLY_CHAIN_ENV_URL", "http://localhost:8000")
MODEL_NAME   = os.getenv("MODEL_NAME", "unsloth/Qwen2.5-7B-Instruct-bnb-4bit")
LOG_INTERVAL = 10
TASKS = ["easy_phase_inventory", "medium_phase_inventory", "hard_phase_inventory"]

SYSTEM_PROMPT = """You are an AI procurement manager for a perishable goods distributor.

You manage inventory over 30 days. Every unit expires 15 days after arrival. Your supplier has hidden loyalty tiers that affect your costs and crisis allocation — infer your tier from observable signals.

Every day make exactly four decisions:
1. action_type: "order" (standard lead time), "emergency_restock" (arrives in 1 day, costs more), or "hold"
2. quantity: how many units to order (null if hold)
3. sell_price: price per unit to customers (affects demand via price elasticity)
4. negotiation_message: professional message to your supplier (builds loyalty over time)

Key signals to watch:
- emergency_surcharge_rate: 2.5x=Gold tier, 3.0x=Silver tier, 4.0x=Bronze tier
- supplier_last_message tone: warm=Gold, neutral=Silver, cold/restrictive=Bronze
- lead_time_accuracy: consistent delays indicate Bronze tier
- recent_neg_scores: your last 3 negotiation scores (0.0–1.0)

On day 21: supply disruption begins — Bronze agents get 50% of ordered quantity, Silver 80%, Gold 100%.

Always respond with exactly one JSON on a single line:
{"action_type": "...", "quantity": <int or null>, "sell_price": <float>, "negotiation_message": "<str>"}"""

print(f"Environment URL : {ENV_URL}")
print(f"Model           : {MODEL_NAME}")

In [ ]:
# Cell 3 — Real environment HTTP client
import json
import requests

class SupplyChainEnvClient:
    """HTTP client for the Adaptive Supply Chain environment server."""

    def __init__(self, base_url=ENV_URL):
        self.base_url = base_url.rstrip("/")

    def reset(self, task="easy_phase_inventory", seed=None):
        payload = {"task": task}
        if seed is not None:
            payload["seed"] = seed
        r = requests.post(f"{self.base_url}/reset", json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()
        obs = data.get("observation", {})
        return obs.get("prompt", str(obs)) if isinstance(obs, dict) else str(obs)

    def step(self, action_str: str):
        action = self._parse_action(action_str)
        r = requests.post(f"{self.base_url}/step", json=action, timeout=30)
        r.raise_for_status()
        data = r.json()
        obs = data.get("observation", {})
        prompt = obs.get("prompt", str(obs)) if isinstance(obs, dict) else str(obs)
        return prompt, data.get("reward", 0.0), data.get("done", False), data.get("info", {})

    @staticmethod
    def _parse_action(raw: str) -> dict:
        clean = raw.strip()
        if "```" in clean:
            parts = clean.split("```")
            clean = parts[1] if len(parts) > 1 else clean
            if clean.startswith("json"):
                clean = clean[4:]
        start, end = clean.find("{"), clean.rfind("}") + 1
        try:
            return json.loads(clean[start:end]) if start != -1 and end > 0 else {}
        except json.JSONDecodeError:
            return {"action_type": "hold", "quantity": None, "sell_price": 265.0, "negotiation_message": ""}


# Smoke test — confirm server is reachable
try:
    client = SupplyChainEnvClient()
    obs = client.reset(task="easy_phase_inventory", seed=0)
    print("Server reachable. First observation (truncated):")
    print(obs[:400])
except Exception as e:
    print(f"Server not reachable: {e}")
    print(f"Make sure the environment is running at {ENV_URL}")

In [ ]:
# Cell 4 — Evaluation function
import sys
import numpy as np
import torch

# graders.py is in the project root — add it to path if needed
if "." not in sys.path:
    sys.path.insert(0, ".")

from graders import PhaseHistory, grade_phase


def evaluate_agent(model, tokenizer, env_client, n_episodes=3, seed=42):
    """Run n_episodes per task and return mean reward and grade against the real server."""
    results = {}

    for task in TASKS:
        phase = task.replace("_phase_inventory", "")  # "easy" | "medium" | "hard"
        ep_rewards, ep_grades = [], []

        for ep in range(n_episodes):
            obs = env_client.reset(task=task, seed=seed + ep)
            done = False
            total_reward = 0.0
            total_cost_accum = 0.0
            demand_h, fulfilled_h, spoilage_h, revenue_h = [], [], [], []
            valid_actions, total_actions = 0, 0

            while not done:
                prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{obs}\n<|assistant|>\n"
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=300,
                        temperature=0.7,
                        do_sample=True,
                    )
                response = tokenizer.decode(
                    out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
                )

                obs, reward, done, info = env_client.step(response)
                total_reward += reward
                total_actions += 1
                if not info.get("action_malformed", False):
                    valid_actions += 1

                rc = info.get("reward_components", {})
                demand_h.append(float(info.get("actual_demand", 80)))
                fulfilled_h.append(float(info.get("units_fulfilled", 80)))
                spoilage_h.append(float(info.get("units_spoiled", 0)))
                revenue_h.append(float(rc.get("gross_profit", 0)))
                total_cost_accum += abs(float(rc.get("order_cost", 0)))

            ep_rewards.append(total_reward)
            history = PhaseHistory(
                phase=phase,
                demand_history=demand_h,
                fulfilled_history=fulfilled_h,
                spoilage_history=spoilage_h,
                revenue_history=revenue_h,
                total_cost=total_cost_accum,
                valid_actions=valid_actions,
                total_actions=total_actions,
            )
            ep_grades.append(grade_phase(history))

        results[task] = {
            "mean_reward": np.mean(ep_rewards),
            "std_reward": np.std(ep_rewards),
            "mean_grade": np.mean(ep_grades),
        }
        print(
            f"{task}: "
            f"reward={results[task]['mean_reward']:.1f}\u00b1{results[task]['std_reward']:.1f} "
            f"| grade={results[task]['mean_grade']:.4f}"
        )

    return results


print("evaluate_agent() ready.")

In [ ]:
# Cell 5 — Load model with Unsloth + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 6 — Baseline evaluation (untrained model)
env_client = SupplyChainEnvClient()

print("=" * 60)
print("BASELINE (UNTRAINED)")
print("=" * 60)
baseline_results = evaluate_agent(model, tokenizer, env_client, n_episodes=3, seed=42)

overall_baseline = np.mean([r["mean_grade"] for r in baseline_results.values()])
print(f"\nOverall baseline grade: {overall_baseline:.4f}")

In [ ]:
# Cell 7 — GRPO training
import datasets
from trl import GRPOConfig, GRPOTrainer


def build_dataset(env_client, n=150):
    """Build a dataset of single-step observations across all three phases."""
    prompts = []
    for i in range(n):
        task = TASKS[i % len(TASKS)]
        obs = env_client.reset(task=task, seed=i)
        prompts.append({
            "prompt": f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{obs}\n<|assistant|>\n",
            "task": task,
        })
    return datasets.Dataset.from_list(prompts)


def reward_fn(prompts, completions, task=None, **kwargs):
    """
    Single-step reward function for GRPO.
    Resets a fresh environment for each (prompt, completion) pair and returns
    the step reward — no multi-step rollout needed.
    """
    rewards = []
    tmp = SupplyChainEnvClient()
    tasks_list = task if task is not None else [TASKS[i % len(TASKS)] for i in range(len(prompts))]
    for i, (_, completion) in enumerate(zip(prompts, completions)):
        t = tasks_list[i] if isinstance(tasks_list, list) else tasks_list
        tmp.reset(task=t, seed=i)
        try:
            _, r, _, _ = tmp.step(completion)
        except Exception:
            r = -10.0  # malformed action or server error
        rewards.append(float(r))
    return rewards


train_ds = build_dataset(env_client, n=150)
reward_log = []

cfg = GRPOConfig(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=LOG_INTERVAL,
    save_steps=100,
    report_to="none",
    max_completion_length=300,
    num_generations=4,
)

trainer = GRPOTrainer(
    model=model,
    args=cfg,
    reward_funcs=reward_fn,
    train_dataset=train_ds,
    tokenizer=tokenizer,
)

_orig_log = trainer.log
def _log_hook(logs):
    if "reward" in logs:
        reward_log.append(logs["reward"])
    _orig_log(logs)
trainer.log = _log_hook

trainer.train()
print("Training complete.")

In [ ]:
# Cell 8 — Post-training evaluation
print("=" * 60)
print("POST-TRAINING")
print("=" * 60)
trained_results = evaluate_agent(model, tokenizer, env_client, n_episodes=3, seed=42)

overall_trained = np.mean([r["mean_grade"] for r in trained_results.values()])

print("\n" + "=" * 60)
print("IMPROVEMENT SUMMARY")
print("=" * 60)
print(f"{'Task':<30} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * 60)
for task in TASKS:
    b = baseline_results[task]["mean_grade"]
    t = trained_results[task]["mean_grade"]
    print(f"{task:<30} {b:>8.4f} {t:>8.4f} {t-b:>+8.4f}")
print("-" * 60)
print(f"{'Overall':<30} {overall_baseline:>8.4f} {overall_trained:>8.4f} {overall_trained-overall_baseline:>+8.4f}")

In [ ]:
# Cell 9 — Save reward curve and grade comparison plots
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.dpi"] = 150
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GRPO training reward over steps
if reward_log:
    axes[0].plot(reward_log, color="#2563eb", linewidth=1.5, alpha=0.8)
else:
    axes[0].text(0.5, 0.5, "No reward data", ha="center", va="center", transform=axes[0].transAxes)
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("Step Reward")
axes[0].set_title("Adaptive Supply Chain — GRPO Training Reward")
axes[0].grid(True, alpha=0.3)

# Right: Before vs after grade comparison
tasks_short = ["Easy", "Medium", "Hard"]
b_grades = [baseline_results[t]["mean_grade"] for t in TASKS]
t_grades = [trained_results[t]["mean_grade"] for t in TASKS]
x, w = range(3), 0.35
axes[1].bar([i - w / 2 for i in x], b_grades, w, label="Untrained", color="#ef4444", alpha=0.8)
axes[1].bar([i + w / 2 for i in x], t_grades, w, label="Trained",   color="#22c55e", alpha=0.8)
axes[1].set_xlabel("Phase")
axes[1].set_ylabel("Grade Score (0.0 \u2013 1.0)")
axes[1].set_title("Grade: Before vs After GRPO Training")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(tasks_short)
axes[1].set_ylim(0, 1.0)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("reward_curves.png", bbox_inches="tight")
plt.savefig("grade_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: reward_curves.png, grade_comparison.png")

In [ ]:
# Cell 10 — Day-21 supply disruption demo
# Shows how trained vs untrained agent responds to the crisis.

def demo_crisis(model_to_test, label=""):
    env = SupplyChainEnvClient()
    obs = env.reset(task="hard_phase_inventory", seed=0)

    # Fast-forward to day 21 with hold actions
    for day in range(1, 21):
        hold_action = json.dumps({
            "action_type": "hold",
            "quantity": None,
            "sell_price": 310.0,
            "negotiation_message": "",
        })
        obs, _, done, _ = env.step(hold_action)
        if done:
            break

    print(f"\n{'='*60}")
    print(f"DAY 21 STATE ({label}):")
    print(obs[:800])
    print(f"{'='*60}")

    prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{obs}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_test.device)

    with torch.no_grad():
        out = model_to_test.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=False,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{label} AGENT RESPONSE:")
    print(response)
    return response


# Run demo with trained model
trained_response = demo_crisis(model, "TRAINED")

# To compare with untrained:
# import copy; baseline_model = copy.deepcopy(model)  # save BEFORE Cell 7
# demo_crisis(baseline_model, "UNTRAINED")

In [ ]:
# Cell 11 — Save trained model
SAVE_PATH = "./checkpoints/supply-chain-rl-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

# Optional: push to HuggingFace Hub
# model.push_to_hub("your-org/adaptive-supply-chain-qwen25-7b")
# tokenizer.push_to_hub("your-org/adaptive-supply-chain-qwen25-7b")